In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import torch

from tsfm_lens.chronos.circuitlens import CircuitLensChronos
# from tsfm_lens.utils import (
#     apply_custom_style,
#     load_dyst_batch,
#     set_seed,
# )

In [ ]:
# read in pickle file
model_name = "chronos-t5-base"
config_key = "rf2_sl16"

induction_results_dir = "../../outputs/rrt_induction_scores"
with open(os.path.join(induction_results_dir, model_name, config_key, "center_scores.pkl"), "rb") as f:
    center_scores = pickle.load(f)
    all_center_scores = center_scores["all_center_scores"]
    center_scores_mean, center_scores_std = center_scores["mean"], center_scores["std"]

with open(os.path.join(induction_results_dir, model_name, config_key, "right_scores.pkl"), "rb") as f:
    right_scores = pickle.load(f)
    all_right_scores = right_scores["all_right_scores"]
    right_scores_mean, right_scores_std = right_scores["mean"], right_scores["std"]

with open(
    os.path.join(induction_results_dir, model_name, config_key, "correct_token_attribution.pkl"),
    "rb",
) as f:
    correct_token_attribution = pickle.load(f)
    all_correct_token_attributions = correct_token_attribution["correct_token_attributions"]
    all_token_attributions = correct_token_attribution["all_token_attributions"]
    correct_token_attribution_mean, correct_token_attribution_std = (
        correct_token_attribution["mean"],
        correct_token_attribution["std"],
    )

with open(os.path.join(induction_results_dir, model_name, config_key, "rrt_vars.pkl"), "rb") as f:
    rrt_vars = pickle.load(f)
    std_factor, overall_scores_mean, overall_scores_std = (
        rrt_vars["std_factor"],
        rrt_vars["overall_scores"][0],
        rrt_vars["overall_scores"][1],
    )

entropy_results_dir = "../../outputs/head_entropy"
with open(os.path.join(entropy_results_dir, "head_entropy_ca.pkl"), "rb") as f:
    head_entropy = pickle.load(f)
    avg_ent_l_h = torch.tensor(head_entropy["avg_ent_l_h"])

In [ ]:
print("=" * 80)
print("DATA SHAPES")
print("=" * 80)
print(f"all_center_scores shape: {all_center_scores.shape}")
print(f"  Description: Center position induction scores")
print(f"  Dimensions: (batch, layers, heads)\n")

print(f"all_right_scores shape: {all_right_scores.shape}")
print(f"  Description: Right position induction scores")
print(f"  Dimensions: (batch, layers, heads)\n")

print(f"all_correct_token_attribution shape: {all_correct_token_attributions.shape}")
print(f"  Description: Attribution scores for correct tokens")
print(f"  Dimensions: (batch, layers, heads)\n")

print(f"all_token_attribution shape: {all_token_attributions.shape}")
print(f"  Description: Attribution scores for all tokens")
print(f"  Dimensions: (batch, layers, heads, tokens)\n")

print(f"avg_ent_l_h shape: {avg_ent_l_h.shape}")
print(f"  Description: Average head entropy")
print(f"  Dimensions: (layers, heads)\n")

# print("=" * 80)
# print("SUMMARY STATISTICS")
# print("=" * 80)
# print(f"Model: {model_name}")
# print(f"Config: {config_key}")
# print(f"Std Factor: {std_factor}\n")

# print(f"Overall Scores:")
# print(f"  Mean: {overall_scores_mean:.4f}")
# print(f"  Std:  {overall_scores_std:.4f}\n")

# print(f"Center Scores: {center_scores_mean.shape} (layers x heads)")
# print(f"  Mean range: [{center_scores_mean.min():.4f}, {center_scores_mean.max():.4f}]")
# print(f"  Std range:  [{center_scores_std.min():.4f}, {center_scores_std.max():.4f}]\n")

# print(f"Right Scores: {right_scores_mean.shape} (layers x heads)")
# print(f"  Mean range: [{right_scores_mean.min():.4f}, {right_scores_mean.max():.4f}]")
# print(f"  Std range:  [{right_scores_std.min():.4f}, {right_scores_std.max():.4f}]\n")

# print(f"Correct Token Attribution: {correct_token_attribution_mean.shape} (layers x heads)")
# print(f"  Mean range: [{correct_token_attribution_mean.min():.4f}, {correct_token_attribution_mean.max():.4f}]")
# print(f"  Std range:  [{correct_token_attribution_std.min():.4f}, {correct_token_attribution_std.max():.4f}]")

In [ ]:
center_scores_mean = all_center_scores.mean(dim=0)
right_scores_mean = all_right_scores.mean(dim=0)

# set induction_scores_mean to be the element-wise maximum of center_scores_mean and right_scores_mean
induction_scores_mean = torch.max(center_scores_mean, right_scores_mean)
correct_token_attribution_mean = all_correct_token_attributions.mean(dim=0)

In [ ]:
relative_attribution_scores = (all_correct_token_attributions - all_token_attributions.mean(dim=-1))/all_token_attributions.std(dim=-1)
relative_attributions_mean = relative_attribution_scores.mean(dim=0)
relative_attributions_mean.shape

In [ ]:
# plot heatmap of induction_scores_mean with y axis being layer and x axis being head index

scores = induction_scores_mean.detach().cpu().numpy()  # [layers, heads]

# robust color scaling (avoids a single large head dominating the colormap)
vmin = float(np.quantile(scores, 0.01))
vmax = float(np.quantile(scores, 0.99))

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(scores, aspect="equal", origin="lower", cmap="Blues", vmin=0, vmax=1)

ax.set_title("Induction scores (mean)")
ax.set_xlabel("Head index")
ax.set_ylabel("Layer")

n_layers, n_heads = scores.shape
ax.set_xticks(np.arange(n_heads))
ax.set_xticklabels(np.arange(1, n_heads + 1))
ax.set_yticks(np.arange(n_layers))
ax.set_yticklabels(np.arange(1, n_layers + 1))

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Induction score")

fig.tight_layout()

# save as induction_mosaic.svg/pdf
out_dir = "../../figs/induction_heads/mosaic"
os.makedirs(out_dir, exist_ok=True)
fig.savefig(os.path.join(out_dir, "mosaic.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(out_dir, "mosaic.svg"), bbox_inches="tight")

plt.show()

# --- Separate heatmap: average head entropy (normalized to [0, 1] for plotting only) ---
ent = avg_ent_l_h.detach().cpu().numpy()  # [layers, heads]
ent_min = float(ent.min())
ent_max = float(ent.max())
denom = ent_max - ent_min
if denom == 0.0:
    ent_norm = np.zeros_like(ent, dtype=np.float64)
else:
    ent_norm = (ent - ent_min) / denom

fig2, ax2 = plt.subplots(figsize=(7, 7))
im2 = ax2.imshow(ent_norm, aspect="equal", origin="lower", cmap="Blues_r", vmin=0.0, vmax=1.0)

ax2.set_title("Average head entropy (normalized)")
ax2.set_xlabel("Head index")
ax2.set_ylabel("Layer")

n_layers_e, n_heads_e = ent_norm.shape
ax2.set_xticks(np.arange(n_heads_e))
ax2.set_xticklabels(np.arange(1, n_heads_e + 1))
ax2.set_yticks(np.arange(n_layers_e))
ax2.set_yticklabels(np.arange(1, n_layers_e + 1))

cbar2 = fig2.colorbar(im2, ax=ax2)
cbar2.set_label("Normalized entropy")

fig2.tight_layout()

# optional save alongside induction mosaic
fig2.savefig(os.path.join(out_dir, "entropy_mosaic_norm.pdf"), bbox_inches="tight")
fig2.savefig(os.path.join(out_dir, "entropy_mosaic_norm.svg"), bbox_inches="tight")

plt.show()

In [ ]:
correct_token_threshold = 2.0
right_score_threshold = 0.3
entropy_threshold = 4.0

In [ ]:
# 3x3 lower-triangular grid for:
#   (1) right_scores_mean
#   (2) correct_token_attribution_mean
#   (3) avg_ent_l_h

def _to_1d_numpy(a):
    if isinstance(a, torch.Tensor):
        return a.detach().cpu().flatten().numpy()
    return np.asarray(a).flatten()

def _minmax_norm_1d(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x_min = float(np.min(x))
    x_max = float(np.max(x))
    denom = x_max - x_min
    if denom == 0.0:
        return np.zeros_like(x, dtype=np.float64)
    return (x - x_min) / denom

# Config: normalize entropy axis to [0, 1] for display
normalize_entropy_for_plot = True

x_right = _to_1d_numpy(induction_scores_mean)
y_attr = _to_1d_numpy(relative_attributions_mean)
z_ent_raw = _to_1d_numpy(avg_ent_l_h)  # [layers, heads] -> flat
z_ent = _minmax_norm_1d(z_ent_raw) if normalize_entropy_for_plot else z_ent_raw

vars_ = [x_right, y_attr, z_ent]
labels = [
    "Avg Induction Scores",
    "Avg Correct Token Attribution Scores",
    "Avg Head Entropy (normalized)" if normalize_entropy_for_plot else "Avg Head Entropy",
]

fig, axes = plt.subplots(3, 3, figsize=(12, 12), sharex="col", sharey=False)

for i in range(3):
    for j in range(3):
        ax = axes[i, j]

        if i == j:
            ax.hist(vars_[i], bins=30, alpha=0.8)
            ax.set_title(labels[i])
            ax.set_ylabel("Count")
            ax.grid(True, alpha=0.3)

        elif i > j:
            ax.scatter(vars_[j], vars_[i], alpha=0.6, s=18)
            # ax.set_title(f"{labels[j]} vs {labels[i]}")
            ax.grid(True, alpha=0.3)

        else:
            ax.axis("off")

        if i == 2 and i > j:
            ax.set_xlabel(labels[j])
        if j == 0 and i > j:
            ax.set_ylabel(labels[i])

plt.tight_layout()
plt.show()

# save figure as pdf and svg
fig.savefig("../../figs/induction_heads/triangle_plot/induction_attribution_entropy.pdf")
fig.savefig("../../figs/induction_heads/triangle_plot/induction_attribution_entropy.svg")

In [ ]:
# proportion of relative_attributions_mean > T_c
num_heads = relative_attributions_mean.numel()
num_heads_above_threshold = (relative_attributions_mean > correct_token_threshold).sum().item()
print("P(relative_attributions_mean > T_c):\n", num_heads_above_threshold / num_heads)

# proportion of right_scores_mean > T_r
num_heads_right_above_threshold = (right_scores_mean > right_score_threshold).sum().item()
print("P(right_scores_mean > T_r):\n", num_heads_right_above_threshold / num_heads)

# proportion of avg_ent_l_h < T_e
num_heads_entropy_below_threshold = (avg_ent_l_h < entropy_threshold).sum().item()
print("P(avg_ent_l_h < T_e):\n", num_heads_entropy_below_threshold / num_heads)

# proportion of relative_attributions_mean > T_c and right_scores_mean > T_r
num_heads_both_above_threshold = (
    (relative_attributions_mean > correct_token_threshold) & (right_scores_mean > right_score_threshold)
).sum().item()
print("P(relative_attributions_mean > T_c, right_scores_mean > T_r):\n", num_heads_both_above_threshold / num_heads)

print("=" * 40)

# proportion of relative_attributions_mean > T_c given right_scores_mean > T_r
num_heads_right_above_threshold = (
    (relative_attributions_mean > correct_token_threshold) & (right_scores_mean > right_score_threshold)
).sum().item()
num_heads_right_condition = (right_scores_mean > right_score_threshold).sum().item()
print(
    "P(relative_attributions_mean > T_c | right_scores_mean > T_r):\n",
    num_heads_right_above_threshold / num_heads_right_condition,
)

# proportion of avg_ent_l_h < T_e given relative_attributions_mean > T_c and right_scores_mean > T_r
num_heads_entropy_below_threshold = (
    (avg_ent_l_h < entropy_threshold)
    & (relative_attributions_mean > correct_token_threshold)
    & (right_scores_mean > right_score_threshold)
).sum().item()
print(
    "P(avg_ent_l_h < T_e | relative_attributions_mean > T_c, right_scores_mean > T_r):\n",
    num_heads_entropy_below_threshold / num_heads_both_above_threshold,
)

# proportion of relative_attributions_mean > T_c and right_scores_mean > T_r given avg_ent_l_h < T_e
num_heads_both_above_threshold = (
    (relative_attributions_mean > correct_token_threshold)
    & (right_scores_mean > right_score_threshold)
    & (avg_ent_l_h < entropy_threshold)
).sum().item()
num_heads_entropy_condition = (avg_ent_l_h < entropy_threshold).sum().item()
print(
    "P(relative_attributions_mean > T_c, right_scores_mean > T_r | avg_ent_l_h < T_e):\n",
    num_heads_both_above_threshold / num_heads_entropy_condition,
)

In [ ]:
# Heatmaps over (correct_token_threshold, entropy_threshold) at fixed right_score_threshold
right_score_threshold = 0.3

# Ensure tensors are on CPU for cheap ops / plotting
rel_attr = relative_attributions_mean.detach().cpu()
right_scores = right_scores_mean.detach().cpu()
avg_ent = avg_ent_l_h.detach().cpu()

# Choose threshold ranges that avoid empty conditioning sets (prevents NaNs/gray regions)
ent_vals = torch.unique(avg_ent.flatten()).sort().values
ent_min = float(ent_vals[0].item())
ent_max = float(ent_vals[-1].item())
ent_denom = ent_max - ent_min if ent_max != ent_min else 1.0

# Force the displayed entropy range to start strictly above 3.0 (in raw entropy units)
entropy_floor = 3.0
ent_candidates = ent_vals[ent_vals > entropy_floor]
entropy_low_raw = float(ent_candidates[0].item()) if ent_candidates.numel() > 0 else float(np.nextafter(entropy_floor, np.inf))
entropy_high_raw = ent_max

# For correct_token_threshold, ensure there is always at least one head with (rel_attr > T_c) and (right_scores > T_r)
mask_right = right_scores > float(right_score_threshold)
rel_attr_right = rel_attr[mask_right] if bool(mask_right.any().item()) else rel_attr
rel_attr_vals = torch.unique(rel_attr_right.flatten()).sort().values
max_rel_attr_given_right = float(rel_attr_vals[-1].item())
if rel_attr_vals.numel() > 1:
    correct_high_data = float(rel_attr_vals[-2].item())  # ensures at least one element strictly greater than T_c
else:
    correct_high_data = float(np.nextafter(max_rel_attr_given_right, -np.inf))

# Respect the original requested lower/upper bounds, but trim the upper bound if it would create empty conditioning
correct_low = 2.0
correct_high = min(5.0, correct_high_data)
if not (correct_high > correct_low):
    correct_high = float(np.nextafter(correct_low, np.inf))

correct_token_thresholds = np.linspace(correct_low, correct_high, 100)  # x-axis

# Build entropy threshold sweep in raw units for computation, but normalize for plotting
entropy_thresholds_raw = np.linspace(entropy_low_raw, entropy_high_raw, 100)  # raw entropy units
entropy_thresholds = (entropy_thresholds_raw - ent_min) / ent_denom             # normalized to [0, 1]

# Pre-allocate probability grids: rows=entropy_thresholds (y), cols=correct_token_thresholds (x)
p_ent_given_attr_right = np.full((len(entropy_thresholds_raw), len(correct_token_thresholds)), np.nan, dtype=np.float64)
p_attr_right_given_ent = np.full((len(entropy_thresholds_raw), len(correct_token_thresholds)), np.nan, dtype=np.float64)

for yi, T_e_raw in enumerate(entropy_thresholds_raw):
    mask_ent = (avg_ent < float(T_e_raw))
    ent_count = int(mask_ent.sum().item())

    for xi, T_c in enumerate(correct_token_thresholds):
        mask_attr_right = (rel_attr > float(T_c)) & (right_scores > float(right_score_threshold))
        attr_right_count = int(mask_attr_right.sum().item())

        # Shared numerator: heads satisfying both conditions
        both_count = int((mask_ent & mask_attr_right).sum().item())

        # Subplot 1: P(avg_ent < T_e | rel_attr > T_c, right_scores > T_r)
        if attr_right_count > 0:
            p_ent_given_attr_right[yi, xi] = both_count / attr_right_count

        # Subplot 2: P(rel_attr > T_c, right_scores > T_r | avg_ent < T_e)
        if ent_count > 0:
            p_attr_right_given_ent[yi, xi] = both_count / ent_count

# --- Heatmaps (y-axis uses normalized entropy threshold in [0, 1]) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
extent = [correct_token_thresholds.min(), correct_token_thresholds.max(), entropy_thresholds.min(), entropy_thresholds.max()]

# Keep NaN coloring in case of edge-case empties, but ranges above should avoid them.
cmap = plt.cm.viridis.copy()
cmap.set_bad(color="lightgray")

# Heatmap 1
im0 = axes[0].imshow(
    np.ma.masked_invalid(p_ent_given_attr_right),
    origin="lower",
    aspect="auto",
    extent=extent,
    vmin=0.0,
    vmax=1.0,
    cmap=cmap,
 )
axes[0].set_title(r"$P(\mathrm{sharp\ head}\mid \mathrm{induction\ head})$")
axes[0].set_xlabel("correct_token_threshold (T_c)")
axes[0].set_ylabel("normalized entropy threshold (T_e)")
cbar0 = fig.colorbar(im0, ax=axes[0])
cbar0.set_label("Probability")

# Heatmap 2
im1 = axes[1].imshow(
    np.ma.masked_invalid(p_attr_right_given_ent),
    origin="lower",
    aspect="auto",
    extent=extent,
    vmin=0.0,
    vmax=1.0,
    cmap=cmap,
 )
axes[1].set_title(r"$P(\mathrm{induction\ head}\mid \mathrm{sharp\ head})$")
axes[1].set_xlabel("correct token threshold")
axes[1].set_ylabel("entropy threshold (normalized)")
cbar1 = fig.colorbar(im1, ax=axes[1])
cbar1.set_label("Probability")

plt.show()

In [ ]:
# Contour plot over (correct_token_threshold, right_score_threshold):
# value = proportion of all heads with
#   relative_attributions_mean > T_c  AND  right_scores_mean > T_r

# Ensure tensors are on CPU
rel_attr = relative_attributions_mean.detach().cpu()
right_scores = right_scores_mean.detach().cpu()

rel_attr_flat = rel_attr.flatten()
right_flat = right_scores.flatten()

# =====================
# Config (edit these)
# =====================
n_Tc = 100  # x-resolution
n_Tr = 100  # y-resolution

# If True, pick ranges from data quantiles; if False, use the manual ranges below.
use_quantile_bounds = False
Tc_quantiles = (0.01, 0.99)
Tr_quantiles = (0.01, 0.99)

# Manual ranges (only used when use_quantile_bounds=False)
Tc_manual = (0.0, 5.0)
Tr_manual = (0.25, 0.8)

# Color scale: choose a robust min/max from the computed probability grid
use_robust_color_scale = True
p_color_quantiles = (0.01, 0.99)  # (low, high) quantiles over p_both
p_color_floor = 0.0               # force vmin >= this (often 0.0)
n_color_levels = 21               # number of filled contour levels
add_contour_lines = True

# Marker for the thresholds used earlier in the notebook
show_threshold_marker = True
marker_color = "red"
marker_size = 140
marker_linewidth = 2.5

# =====================
# Range selection
# =====================
rel_np = rel_attr_flat.numpy()
right_np = right_flat.numpy()

if use_quantile_bounds:
    Tc_low, Tc_high = (float(np.quantile(rel_np, Tc_quantiles[0])), float(np.quantile(rel_np, Tc_quantiles[1])))
    Tr_low, Tr_high = (float(np.quantile(right_np, Tr_quantiles[0])), float(np.quantile(right_np, Tr_quantiles[1])))
else:
    Tc_low, Tc_high = map(float, Tc_manual)
    Tr_low, Tr_high = map(float, Tr_manual)

# Guard against degenerate ranges
if not (Tc_high > Tc_low):
    Tc_high = float(np.nextafter(Tc_low, np.inf))
if not (Tr_high > Tr_low):
    Tr_high = float(np.nextafter(Tr_low, np.inf))

correct_token_thresholds = np.linspace(Tc_low, Tc_high, n_Tc)  # x-axis
right_score_thresholds = np.linspace(Tr_low, Tr_high, n_Tr)    # y-axis

# =====================
# Compute grid
# =====================
# Grid: rows=right_score_thresholds (y), cols=correct_token_thresholds (x)
p_both = np.zeros((len(right_score_thresholds), len(correct_token_thresholds)), dtype=np.float64)
num_heads = float(rel_attr_flat.numel())

for yi, T_r in enumerate(right_score_thresholds):
    for xi, T_c in enumerate(correct_token_thresholds):
        mask = (rel_attr_flat > float(T_c)) & (right_flat > float(T_r))
        p_both[yi, xi] = float(mask.sum().item()) / num_heads

# =====================
# Color scaling (data-driven)
# =====================
p_flat = p_both.ravel()
p_min = float(np.nanmin(p_flat))
p_max = float(np.nanmax(p_flat))

if use_robust_color_scale:
    p_lo = float(np.quantile(p_flat, p_color_quantiles[0]))
    p_hi = float(np.quantile(p_flat, p_color_quantiles[1]))
else:
    p_lo, p_hi = p_min, p_max

p_lo = max(float(p_color_floor), p_lo)
if not (p_hi > p_lo):
    p_hi = float(np.nextafter(p_lo, np.inf))

levels = np.linspace(p_lo, p_hi, n_color_levels)

# =====================
# Plot (contour)
# =====================
fig, ax = plt.subplots(figsize=(7.5, 6.0), constrained_layout=True)

cf = ax.contourf(
    correct_token_thresholds,
    right_score_thresholds,
    p_both,
    levels=levels,
    cmap="viridis",
    extend="max",
 )

if add_contour_lines:
    ax.contour(
        correct_token_thresholds,
        right_score_thresholds,
        p_both,
        levels=levels,
        colors="k",
        linewidths=0.3,
        alpha=0.35,
    )

# Red cross at the thresholds used earlier in the notebook
if show_threshold_marker:
    try:
        Tc0 = float(correct_token_threshold)
        Tr0 = float(right_score_threshold)
        ax.scatter(
            [Tc0],
            [Tr0],
            marker="x",
            c=marker_color,
            s=marker_size,
            linewidths=marker_linewidth,
            label="Chosen thresholds",
            zorder=10,
        )
        ax.legend(loc="best", frameon=True)
        if not (Tc_low <= Tc0 <= Tc_high and Tr_low <= Tr0 <= Tr_high):
            print(
                "WARNING: (correct_token_threshold, right_score_threshold) is outside the current plot ranges: ",
false
            )
            print(f"  thresholds: (T_c={Tc0:.3g}, T_r={Tr0:.3g})")
            print(f"  x-range: [{Tc_low:.3g}, {Tc_high:.3g}], y-range: [{Tr_low:.3g}, {Tr_high:.3g}]")
    except NameError:
        print("WARNING: correct_token_threshold/right_score_threshold not defined; skipping marker.")

ax.set_title("P(head above both thresholds)")
ax.set_xlabel("Correct token threshold (T_c)")
ax.set_ylabel("Right score threshold (T_r)")

cbar = fig.colorbar(cf, ax=ax)
cbar.set_label("Proportion of heads")

# Make the scale explicit in case we're using robust quantiles
cbar.ax.set_title(f"[{p_lo:.3f}, {p_hi:.3f}]", fontsize=9)

plt.show()

# Optional save
out_dir_thr = "../../figs/induction_heads/right_correct_contour"
os.makedirs(out_dir_thr, exist_ok=True)
fig.savefig(os.path.join(out_dir_thr, "p_heads_above_Tc_Tr_contour.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(out_dir_thr, "p_heads_above_Tc_Tr_contour.svg"), bbox_inches="tight")